In [ ]:
path_prefix = '../..'

import sys
sys.path.append(path_prefix)

import os
import json
from collections import Counter
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from egh_vlm.utils import load_phd_dataset

## Analysis

In [ ]:
dataset_path = f'{path_prefix}/data/phd/prototype/phd_sample_qwen3_vl_2b.json'
img_folder_path = f'{path_prefix}/data/phd/images'

dataset = load_phd_dataset(dataset_path, img_folder_path)
df = pd.DataFrame(dataset)
df.head()

In [ ]:
image_paths = df['image_path'].to_list()
print(f'Verify image paths')

for image_path in image_paths:
    if not os.path.exists(image_path):
        print(f'Not found: {image_path}')

In [ ]:
label_counts = df['label'].value_counts()
label_counts

In [ ]:
answers = df['answer'].to_list()
answer_legnths = [len(data.split(' ')) for data in answers]
answer_legnths_counts = Counter(answer_legnths)

print(f'answes lengths mean: {round(np.mean(answer_legnths), 2)}')
print(f'answer lengths median: {round(np.median(answer_legnths), 2)}')
print(f'answer lengths q1: {round(np.quantile(answer_legnths, 0.25), 2)}')
print(f'answer lengths q3: {round(np.quantile(answer_legnths, 0.75), 2)}')

plt.figure()
plt.bar(answer_legnths_counts.keys(), answer_legnths_counts.values())
plt.xlabel('Length of answer')
plt.ylabel('Frequency')
plt.title('Distribution of answer str lengths')
plt.show()

## Dataset Construction

In [ ]:
file_path = f'{path_prefix}/data/phd/prototype/phd_sample_qwen3_vl_2b.json'

with open(file_path, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

In [ ]:
def get_balanced_raw_dataset(dataset, seed=42):
    random.seed(seed)
    label_counts = Counter(item['hallucinated_label'] for item in dataset)
    print('Label distribution before balancing:\n', label_counts)

    min_count = min(label_counts.values())
    print(f'Max balanced count per label: {min_count}')

    balanced_dataset = []

    for label in label_counts:
        label_samples = [item for item in dataset if item['hallucinated_label'] == label]
        balanced_dataset.extend(random.sample(label_samples, min_count))
    random.shuffle(balanced_dataset)
    print(f'Balanced dataset size: {len(balanced_dataset)}')
    print('Label distribution after balancing:\n', Counter(item['hallucinated_label'] for item in balanced_dataset))
    print('Task distribution after balancing:\n', Counter(item['task'] for item in balanced_dataset))
    return balanced_dataset

balanced_dataset = get_balanced_raw_dataset(dataset)

In [ ]:
df_balanced = pd.DataFrame(balanced_dataset)
df_balanced.head()

In [ ]:
save_path = f'{path_prefix}/data/phd/prototype/phd_sample_qwen3_vl_2b_balanced.json'
output = df_balanced.to_dict('records')

with open(save_path, 'w') as f:
    json.dump(output, f, indent=4)